In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\Text-Summarizer-project_01\\Text-Summarizer-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\Text-Summarizer-project_01\\Text-Summarizer-project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [6]:
import sys
print(sys.executable)

c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\venv\Scripts\python.exe


In [7]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config
    

In [9]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        print("SOURCE URL =", self.config.source_URL)
        print("LOCAL FILE =", self.config.local_data_file)

        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(
                f"{filename} downloaded successfully!\nDetails:\n{headers}"
            )

        else:
            logger.info(
                f"File already exists with size: {get_size(Path(self.config.local_data_file))}"
            )

    def extract_zip_file(self):
        """
        Extract the downloaded zip file into the unzip directory.
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

        logger.info(f"Zip file extracted to: {unzip_path}")

In [11]:
import requests

url = "https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip"

response = requests.get(url)

print(response.status_code)

200


In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-08-07 21:48:21,686: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-07 21:48:21,686: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-07 21:48:21,686: INFO: common: created directory at: artifacts]
[2026-08-07 21:48:21,691: INFO: common: created directory at: artifacts/data_ingestion]
SOURCE URL = https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
LOCAL FILE = artifacts/data_ingestion/data.zip
[2026-08-07 21:48:21,693: INFO: 3006702425: File already exists with size: ~7718 KB]
[2026-08-07 21:48:21,849: INFO: 3006702425: Zip file extracted to: artifacts/data_ingestion]
